In [2]:
import os
from pathlib import Path

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd

In [3]:
data_path = Path(os.environ["DATA_PATH"])
scripts_path = Path("./scripts")

In [4]:
with (scripts_path / "01_registro.sql").open() as f:
    registro_query = f.read()
    duckdb.sql(registro_query)

df_orig = (
    pd.read_parquet("data/processed/choiceset.parquet")
    .dropna(subset=["partida_main"])
    .assign(geometry=lambda df: gpd.points_from_xy(df["longitud"], df["latitud"]))
    .drop(columns=["longitud", "latitud"])
    .pipe(gpd.GeoDataFrame, geometry="geometry", crs="EPSG:4326")
    .to_crs("EPSG:6372")
)

IOException: IO Error: Cannot open file "/Users/rodolfofigueroa/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/mexicali_data/processing/RPPC_vivienda-interes-social-nueva_2020-2024.xlsx": No such file or directory

LINE 55:         FROM read_xlsx(
                      ^

In [5]:
df_orig

NameError: name 'df_orig' is not defined

In [41]:
df_init = (
    pd.read_excel(
        data_path / "processing" / "RPPC_vivienda-interes-social-nueva_2020-2024.xlsx",
    )
    .rename(
        columns={
            "Inmobiliaria": "inmobiliaria",
            "Lote": "lote",
            "Manzana": "manzana",
            "Fraccionamiento": "fraccionamiento",
            "Direccion": "direccion",
            "Razón social": "razon_social",
            "Comprador": "comprador",
            "Acreedor": "acreedor",
            "Acreedores": "acreedores",
            "Tipo de vivienda": "tipo_vivienda",
            "Categoría": "categoria",
            "Folio real": "folio",
            "Fecha de operacion": "fecha_operacion",
            "Competencia actual": "competencia_actual",
            "Valor de operacion": "valor_operacion",
            "Monto de crédito": "monto_credito",
            "Fecha de partida": "fecha_partida",
            "Partida": "partida",
            "Superficie": "mts_superficie",
            "Edad": "edad",
            "Latitud": "latitud",
            "Longitud": "longitud",
            "Mercado Exe": "mercado_exe",
        },
    )
    .assign(
        folio=lambda df: pd.to_numeric(df["folio"], errors="coerce").astype("Int64"),
        fecha_operacion=lambda df: pd.to_datetime(df["fecha_operacion"]),
        competencia_actual=lambda df: (
            df["competencia_actual"].str.lower().eq("sí").astype("Int64")
        ),
        valor_operacion=lambda df: pd.to_numeric(
            df["valor_operacion"],
            errors="coerce",
        ),
        monto_credito=lambda df: pd.to_numeric(df["monto_credito"], errors="coerce"),
        fecha_partida=lambda df: pd.to_datetime(df["fecha_partida"]),
        partida=lambda df: pd.to_numeric(df["partida"], errors="coerce").astype(
            "Int64",
        ),
        mts_superficie=lambda df: pd.to_numeric(df["mts_superficie"], errors="coerce"),
        edad=lambda df: pd.to_numeric(df["edad"], errors="coerce").astype("Int64"),
        latitud=lambda df: pd.to_numeric(df["latitud"], errors="coerce"),
        longitud=lambda df: pd.to_numeric(df["longitud"], errors="coerce"),
        mercado_exe=lambda df: df["mercado_exe"].str.lower().eq("sí"),
    )
    .assign(
        fraccionamiento=lambda df: (
            df["fraccionamiento"]
            .str.strip()
            .replace(
                {
                    r"^FRACCIONAMIENTO VILLANOVA.*": "VILLANOVA",
                    r".*PORTICOS DEL VALLE.*": "PORTICOS DEL VALLE",
                },
                regex=True,
            )
            .where(
                lambda x: ~x.str.startswith("FRAC"),
                lambda x: (
                    x.str.replace("FRACCIONAMIENTO", "")
                    .str.replace("FRACC", "")
                    .str.replace("FRACCTO", "")
                    .str.replace("FRACTO", "")
                    .str.strip()
                ),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA BALBUENA"),
                lambda x: x.str.replace("DE MEXICALI BC DENTRO DE LA", "").str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA GRANJAS AGRICOLAS"),
                lambda x: x.str.replace(
                    "FOLIO REAL 1598810 DE MEXICALI BC",
                    "",
                ).str.strip(),
            )
        ),
        coords=lambda df: df["fraccionamiento"].map(
            {
                "Angeles De Puebla": (32.564770, -115.339935),
                "COLONIA BALBUENA": (32.630970, -115.472679),
                "Corceles Residencial": (32.567668, -115.469794),
                "Gran Foresta": (32.563446, -115.434414),
                "Huertas del Colorado": (32.563662, -115.411109),
                "LA RIOJA SECCION CASTILLAUNA": (32.656198, -115.364974),
                "PORTICOS DEL VALLE": (32.595718, -115.438633),
                "Parajes De Puebla": (32.557625, -115.346180),
                "Quinta Granada": (32.572234, -115.469416),
                "Quinta Granada 3": (32.567764, -115.473000),
                "San Andres": (32.577850, -115.424811),
                "Valle Oriente": (32.571884, -115.361330),
                "Privadas Condesa": (32.595546, -115.344834),
            },
        ),
        new_lat=lambda df: df["coords"].apply(lambda x: x[0] if not pd.isna(x) else np.nan),
        new_lon=lambda df: df["coords"].apply(lambda x: x[1] if not pd.isna(x) else np.nan),
        latitud=lambda df: df["latitud"].fillna(df["new_lat"]),
        longitud=lambda df: df["longitud"].fillna(df["new_lon"]),
        privada=lambda df: (
            df["direccion"]
            .str.strip()
            .where(
                lambda x: ~x.str.startswith("DESARROLLO URBANO LA CONDESA"),
                lambda x: x.str.replace(
                    r"DESARROLLO URBANO LA CONDESA SECCI(O|Ó)N",
                    "WANTED_",
                    regex=True,
                ).str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith(
                    "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                ),
                lambda x: x.str.replace(
                    "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                    "WANTED_VICTORIA",
                ).str.strip(),
            )
            .where(lambda x: x.str.startswith("WANTED_"), None)
            .str.replace("WANTED_", "")
            .str.strip()
        ),
        direccion=lambda df: df["direccion"]
        .str.replace("LOCALIZACION:", "")
        .str.strip(),
    )
    .drop(columns=["coords", "new_lat", "new_lon"])
    .query("~fraccionamiento.str.match(r'.*PUERTO DE SAN FELIPE.*')")
)

In [ ]:
df_init[]

,inmobiliaria,folio,fecha_operacion,Municipio,lote,manzana,fraccionamiento,direccion,razon_social,comprador,...,partida,mts_superficie,acreedores,tipo_vivienda,edad,latitud,longitud,mercado_exe,categoria,privada
0,Brasa,1538428,2021-02-11,MEXICALI,NaN,NaN,LA RIOJA SECCION CASTILLAUNA,NaN,PARCELAS CHUVISCAR,INDIVIDUO,...,5921278,NaN,NaN,CONDOMINIO,27,32.656198,-115.364974,False,Competencia inmobiliaria,NaN
1,Brasa,1538428,2020-04-20,MEXICALI,NaN,NaN,LA RIOJA SECCION CASTILLAUNA,NaN,BRASA,PARCELAS CHUVISCAR,...,5898811,NaN,NaN,CONDOMINIO,<NA>,32.656198,-115.364974,False,Competencia inmobiliaria,NaN
2,Casas Cadena,334547,2024-12-30,MEXICALI,39,2,Valle Del Progreso,FRACCIONAMIENTO VALLE DEL PROGRESO,INMOBILIARIA Y FRACCIONADORA CADENA,INDIVIDUO,...,6051425,120.000,INFONAVIT,VIVIENDA,22,32.588319,-115.579407,False,Competencia inmobiliaria,NaN
3,Casas Cadena,334582,2024-12-23,MEXICALI,4,2,Valle Del Progreso,FRACCIONAMIENTO VALLE DEL PROGRESO,INMOBILIARIA Y FRACCIONADORA CADENA,INDIVIDUO,...,6049768,120.000,INFONAVIT,VIVIENDA,23,32.588319,-115.579407,False,Competencia inmobiliaria,NaN
4,Casas Cadena,334612,2024-12-23,MEXICALI,8,1,Valle Del Progreso,FRACCIONAMIENTO VALLE DEL PROGRESO,INMOBILIARIA Y FRACCIONADORA CADENA,INDIVIDUO,...,6049447,120.000,INFONAVIT,VIVIENDA,25,32.588319,-115.579407,False,Competencia inmobiliaria,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7824,Spinm,1545204,2020-01-08,MEXICALI,2 FRACCION 8,32,NaN,FRACCIONAMIENTO RESIDENCIAL QUINTA DEL REY,SPINM,INDIVIDUO,...,5905051,137.400,INFONAVIT,VIVIENDA,<NA>,NaN,NaN,False,Competencia inmobiliaria,NaN
7825,Urbi,7955,2023-05-03,MEXICALI,32,3,NaN,FRACCIONAMIENTO SAN PEDRO RESIDENCIAL,URBI DESARROLLOS URBANOS,INDIVIDUO,...,5985054,507.672,NaN,VIVIENDA,53,NaN,NaN,False,Competencia inmobiliaria,NaN
7826,Urbi,7954,2023-05-03,MEXICALI,31,3,NaN,FRACCIONAMIENTO SAN PEDRO RESIDENCIAL,URBI DESARROLLOS URBANOS,INDIVIDUO,...,5984943,500.000,NaN,VIVIENDA,53,NaN,NaN,False,Competencia inmobiliaria,NaN
7827,Urbi,188188,2021-08-12,MEXICALI,27,6,NaN,FRACCIONAMIENTO MONTECARLO RESIDENCIAL,URBI DESARROLLOS URBANOS,INDIVIDUO,...,5936683,120.015,NaN,VIVIENDA,59,NaN,NaN,False,Competencia inmobiliaria,NaN


In [ ]:
df_init["geometry_existing"]

0                   POINT (NaN NaN)
1                   POINT (NaN NaN)
2       POINT (-115.57941 32.58832)
3       POINT (-115.57941 32.58832)
4       POINT (-115.57941 32.58832)
                   ...             
7824                POINT (NaN NaN)
7825                POINT (NaN NaN)
7826                POINT (NaN NaN)
7827                POINT (NaN NaN)
7828                POINT (NaN NaN)
Name: geometry_existing, Length: 7812, dtype: geometry